In [ ]:
# '''保留用于测试用的, Score转换策略1描述: 拓扑是否优先进行变换'''
# from PIL import Image
# from matplotlib.colors import ListedColormap
# import cc3d
# from matplotlib import colors
# import random
# ''' 具体如何选 '''
# def select_area_topoPrior(x1, x2, s, delta_lamda):
#     # 找出 x2 中相较于 x1 的新增区域
#     x_delta = np.copy(x2)
#     x_delta[x1 == 1] = 0  # 将 x1 中已存在的部分置为 0，仅保留新增部分

#     # # 绘制新增区域
#     # fig, ax = plt.subplots(figsize=(10, 6))
#     # ax.imshow(x_delta, cmap='gray')  # 使用灰度显示新增区域
#     # ax.axis('off')
#     # ax.set_title("新增区域")
#     # plt.show()

#     # x_delta 与 x1 求邻域的集合
#     x_delta_1 = np.copy(x_delta)
#     for i in range(1, x_delta_1.shape[0] - 1):  # 遍历行（避免边界点）
#         for j in range(1, x_delta_1.shape[1] - 1):  # 遍历列（避免边界点）
#             if x_delta_1[i, j] == 1:
#                 flag = False
#                 neighbors = [(i + dx, j + dy) for dx in [-1, 0, 1] for dy in [-1, 0, 1] if (dx, dy) != (0, 0)]
#                 for ni, nj in neighbors:
#                     if x1[ni, nj] == 1:  # 如果邻居在 x1 中为 1
#                         flag = True
#                         break
#                 # 如果没有邻居在 x1 中为 1，则将 x_delta_1 中该点设置为 0
#                 if not flag:
#                     x_delta_1[i, j] = 0  # 注意这里的赋值运算

#     # 绘制处理后的新增区域
#     # fig, ax = plt.subplots(figsize=(10, 6))
#     # ax.imshow(x_delta_1, cmap='gray')  # 使用灰度显示处理后的新增区域
#     # ax.axis('off')
#     # ax.set_title("新增区域2")
#     # plt.show()
        
#     # 拓展x_delta_1
#     x_delta_2 = np.copy(x_delta_1)
#     change = True
#     while change:
#         change = False
#         new_delta_2 = np.copy(x_delta_2)
#         for i in range(1, x_delta.shape[0] - 1):
#             for j in range(1, x_delta.shape[1] - 1):
#                 # 检查当前点是否属于 x_delta 且周围是否有在 x_delta_2 中的邻居
#                 if x_delta[i, j] == 1 and x_delta_2[i, j] == 0:
#                     neighbors = [(i + dx, j + dy) for dx in [-1, 0, 1] for dy in [-1, 0, 1] if (dx, dy) != (0, 0)]
#                     if any(x_delta_2[ni, nj] == 1 for ni, nj in neighbors):
#                         new_delta_2[i, j] = 1
#                         change = True
#         x_delta_2 = new_delta_2  # 更新 x_delta_2
        
#     # 绘制扩展后的区域
#     # fig, ax = plt.subplots(figsize=(10, 6))
#     # ax.imshow(x_delta_2, cmap='gray')  # 使用灰度显示扩展后的区域
#     # ax.axis('off')
#     # ax.set_title("扩展后的新增区域")
#     # plt.show()
    
#     # 新增的已完成
#     x2_topo = np.copy(x1)
#     x2_topo[x_delta_2 == 1] = 1
    
#     # score 后移动
#     # 获取x_delta_3
#     x_delta_3 = np.copy(x_delta)
#     x_delta_3[x_delta==x_delta_2] = 0
#     # fig, ax = plt.subplots(figsize=(10, 6))
#     # ax.imshow(x_delta_3, cmap='gray')  # 使用灰度显示扩展后的区域
#     # ax.axis('off')
#     # ax.set_title("x_delta_3")
#     # plt.show()

#     # 将 2D 图像扩展为 3D，以适应 cc3d.connected_components 的 3D 连通性
#     image_3d = np.expand_dims(x_delta_3, axis=0)  # 变为 (1, H, W)
#     # 使用 cc3d.connected_components 获取 8 连通分量
#     labels_out, numcomp = cc3d.connected_components(image_3d, connectivity=26, return_N=True)
#     print(numcomp) # 52个
#     # 由于输入是单层图像，输出 labels_out 也会是单层 (1, H, W)
#     x_delta_3_components_array = labels_out[0]  # 将结果恢复为 2D
    
#     # 对x_delta_3进行score变换, 有一个假设, 就是已经是x2_topo一定是不能选择进来的,因为其一定大于其他区域值, 所以要遮挡
    
#     x2_topo # 进行遮挡的部分 x2_topo == x2 + Xtopo
#     s # 进行变换的score值
#     x_delta_3_components_array # 先进行扩充, 再寻找边界框内部除了遮挡的最大值,对s中过这个连通分量位置的整体进行替换值. 如果找不到则每个值-delta_lamda
#     mins = [] # 记录所有的最小
#     for i in range(numcomp):
#         comps_coordinates = [] 
#         for m in range(x_delta_3_components_array.shape[0]):
#             for n in range(x_delta_3_components_array.shape[1]):
#                 if(x_delta_3_components_array[m,n]==i+1):
#                     mins.append(x_delta_3_components_array[m,n])
#                     comps_coordinates.append((m,n))
#                     dx,dy = (2,2) # 4邻域,距离为2
#                     # 记录最大值
#                     num_l = [s[m+xl, n+yl] for xl in [-2, -1, 1, 2] for yl in [-2, -1, 1, 2] if x_delta_3_components_array[xl, yl]!= i+1 and x2_topo[xl, yl] == False] # 不要是自己, 并且要用x2_topo进行遮挡
#                     maxs = num_l
#         min_mins = min(mins) 
#         filtered_maxs = [x for x in maxs if x < min_mins] 
#         # 检查是否存在符合条件的元素
#         if filtered_maxs:
#             result = max(filtered_maxs)  # 找到符合条件的最大值
#             for (m,n) in comps_coordinates:
#                 s[m,n] = result  # 找到符合条件的最大值
#         else:
#             result = None  # 如果没有符合条件的值
#             for (m,n) in comps_coordinates:
#                 s[m,n] = s[m,n] - delta_lamda 
                
#     # 返回处理后的(新增的中间结果, 处理的score)
#     return x2_topo, s

# def first_mask_topo_process(mask, remove_min_size):
#     labels_out, numcomp = cc3d.connected_components(mask, connectivity=8, return_N=True)
#     unique_labels, counts = np.unique(labels_out, return_counts=True)
#     remove_labels = unique_labels[counts < remove_min_size]
#     mask[np.isin(labels_out, remove_labels)] = 0
#     return mask
    
# predicted_mask_n = val_scores[:, :, 1] > 0.9  
# predicted_mask_n = first_mask_topo_process(predicted_mask_n,remove_min_size = 10)
# predicted_mask_n1 = val_scores[:, :, 1] > 0.89  
# x2_topo, update_s = select_area_topoPrior(predicted_mask_n, predicted_mask_n1, val_scores[:, :, 1], delta_lamda=0.01)

# # 绘制
# fig, ax = plt.subplots(1, 3, figsize=(15, 10))
# ax[0].imshow(predicted_mask_n, cmap='gray')
# ax[0].axis('off')
# ax[1].imshow(predicted_mask_n1, cmap='gray')
# ax[1].axis('off')
# ax[2].imshow(x2_topo, cmap='gray')
# ax[2].axis('off')
# plt.show()

    
# ''' 迭代进行score变换 '''
# def score_transform_topo(lamda_n, delta_lamda, n_iteration):
#   if(lamda_n <= 0):
#     return
#   lamda_n = lamda_n
#   delta_lamda = delta_lamda
#   for i in range(n_iteration):
#     lamda_n1 = lamda_n - delta_lamda
#     predicted_mask_n = val_scores[:, :, i] > lamda_n  # 应用阈值 lamhat 生成二值掩码
#     predicted_mask_n1 = val_scores[:, :, i] > lamda_n1  # 应用阈值 lamhat 生成二值掩码

#     # 根据拓扑进行优先选择
#     topo_predicted_mask_n1 = select_area_topoPrior(predicted_mask_n, predicted_mask_n1)
    
    
#     # 创建一个三通道的彩色图像来显示对比结果
#     comparison_image = np.zeros((*topo_predicted_mask_n1.shape, 3), dtype=np.uint8)
#     comparison_image[predicted_mask_n == 1] = [255, 255, 255] 
#     new_in_n1 = (topo_predicted_mask_n1 == 1) & (predicted_mask_n == 0)
#     comparison_image[new_in_n1] = [0, 255, 0]  # 绿色
#     # 绘制
#     fig, ax = plt.subplots(1, 2, figsize=(18, 10))
#     ax[0].imshow(predicted_mask_n, cmap='gray')
#     ax[0].axis('off')
#     ax[1].imshow(comparison_image)
#     ax[1].axis('off')
#     plt.show()
#     # 继续迭代
#     lamda_n = lamda_n1
#     predicted_mask_n = topo_predicted_mask_n1
#     predicted_mask_n1 = topo_predicted_mask_n1


# # score_transform_topo(lamda_n = 0.9,delta_lamda=0.1,n_iteration=10)